# 가까운 쪽 무릎·발목 추가 판독

학습 4장, 검증 우선 3장과 예비 1장을 원본에서 골랐다. 전신 박스를 그리고 선수 본인의 가까운 쪽 **무릎 관절선과 발목 복사뼈**를 표시한다. ‘우선’은 눈으로 먼저 살펴볼 순서다. 직접 보이는지는 직접 찍을 때 정한다.

**모든 셀 실행** 후 3단계에서 시작한다. 1차 이벤트 라벨이 없는 장면은 표에 ‘원본 영상 육안 확인’으로 표시한다.

### 1. 선수와 프레임 확인

In [ ]:
import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import display

from wlcv.annotation.knee_ankle_candidates import JOINTS, REVIEW_FOLDER, load_candidates, review_status
from wlcv.annotation.performer_box import PerformerBox
from wlcv.annotation.pose_points import pose_point_editor
from wlcv.config import ROOT, VIDEOS
from wlcv.data.frames import read_rgb

frames = load_candidates()
names = {"split": "분할", "video_id": "영상", "frame": "원본 프레임", "side": "가까운 쪽",
         "phase": "동작 시점", "basis": "시점 근거", "priority": "순서", "cue": "원본에서 볼 부분"}
display(frames[list(names)].rename(columns=names))

### 2. 원본 미리보기

윗줄은 학습, 아랫줄은 검증이다. 검증의 a05 41번은 오른쪽 사례지만 보호대와 테이핑이 있어 예비로 두었다. 양말이나 신발의 접힌 부분과 복사뼈를 구분하기 어려우면 3단계에서 추정 또는 판독 불가로 기록한다.

In [ ]:
groups = list(frames.groupby("split", sort=False))
columns = max(len(group) for _, group in groups)
fig, axes = plt.subplots(len(groups), columns, figsize=(3 * columns, 7), squeeze=False, layout="constrained")
for row, (split, group) in enumerate(groups):
    for ax in axes[row]:
        ax.axis("off")
    for ax, scene in zip(axes[row], group.itertuples()):
        ax.imshow(read_rgb(VIDEOS / scene.filename, scene.frame))
        ax.set_title(f"{split} · {scene.person} #{scene.frame} · {scene.side}")

preview = ROOT / "outputs/pose_baseline/knee_ankle_candidates/preview.jpg"
preview.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(preview, dpi=140)
plt.show()

### 3. 한 장씩 판독하기

1. **장면**에서 프레임을 고른 뒤 **수행자 박스**에서 머리부터 발끝까지 감싸는 대각선 두 모서리를 차례로 클릭한다. 두 번째 클릭에 박스가 저장된다.
2. **무릎**을 선택하고 표에 적힌 쪽의 관절선을 클릭한다. **발목**을 선택하고 같은 다리의 복사뼈를 클릭한다. 클릭할 때마다 원본 픽셀 좌표가 저장된다. 작은 관절은 **원본 크기로 확대**해 확인한다.
3. 옷이나 보호대에 가렸지만 위치를 짚을 수 있으면 **찍을 점**을 추정으로 바꾼 뒤 클릭한다. 위치를 짚기 어려우면 **가려서 판독 불가** 또는 **위치 불확실**을 누른다. 이때는 좌표 없이 상태가 저장된다.
4. 수정하려면 같은 항목을 다시 클릭한다. **다시 읽기**로 확인한 뒤 다음 장면을 선택한다.

In [ ]:
status = review_status(frames)
pending = status.index[~status["판독 완료"] | status["두 직접점 모두 박스 안"].eq("박스·점 재확인")]
choices = [(f"{r.split} · {r.person} {r.frame}번 · {r.phase} · {r.priority}", r.Index)
           for r in frames.itertuples()]
selector = widgets.Dropdown(options=choices, value=pending[0] if len(pending) else 0,
                            description="장면", layout=widgets.Layout(width="min(100%, 620px)"))
panel = widgets.VBox()
cue = widgets.Label()

def open_frame(change=None):
    scene = frames.loc[selector.value]
    dest = REVIEW_FOLDER / scene.split
    box = PerformerBox(dest, scene.video_id, scene.frame)
    cue.value = scene.cue
    panel.children = (pose_point_editor(
        VIDEOS / scene.filename, scene.frame, scene.person, scene.side,
        split=scene.split, annotations=dest, joints=JOINTS, trunk_review=True, performer_box=box,
        frame_label=f"{scene.video_id} · {scene.frame}번 · {scene.phase}",
    ),)

selector.observe(open_frame, names="value")
open_frame()
display(selector, cue, panel)

### 4. 저장 결과 확인

판독 후 이 셀을 다시 실행한다. ‘판독 완료’는 박스와 두 관절의 상태가 모두 저장됐다는 뜻이다. 두 관절을 모두 직접 확인하고 박스 안에 찍었을 때만 마지막 열에 ‘예’가 나온다. 박스와 관절은 `data/annotations/knee_ankle_candidates/` 아래 학습·검증 폴더에 따로 저장된다.

In [ ]:
saved = review_status(frames)
display(saved)
for split, group in saved.groupby("분할", sort=False):
    done = group["판독 완료"].sum()
    pairs = group["두 직접점 모두 박스 안"].eq("예").sum()
    print(f"{split} · 판독 완료 {done}/{len(group)}장 · 박스 안의 두 직접점 {pairs}장")